# Chapter 12 — SOAR와 LLM Agent (실습)

> **Chapter 12 · Part II**

Mock 환경에서만 동작 (실제 IP 차단 X).

## 실습 목표
1. 4 mock tool 정의 + 정책 게이트(화이트리스트·고위험 승인)
2. Agent loop
3. 5 시나리오 — Brute Force / SQLi / DDoS + **차단하면 안 되는 사례**(사내 스캐너·단발 실패)
4. 고위험 도구 승인 게이트 — 승인 없이는 실행되지 않음
5. dry-run vs 실행(execute) 대비
6. 오염된 tool 결과(§12.4.3) 재현 — delimiter 유무 비교

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q pandas

In [ ]:
import os, json, re, html
API_KEY = os.environ.get('ANTHROPIC_API_KEY')
USE_REAL = bool(API_KEY)
if USE_REAL:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

---
## Step 1. Tool 정의 + 정책 게이트가 있는 Mock 실행기

`execute_tool` 은 단순 실행기가 아니라 **정책 게이트**를 함께 가집니다. (1) 화이트리스트/
사내망 IP 는 `block_ip` 를 거부하고, (2) 고위험 도구(`block_ip` 등)는 실행 모드에서 승인이
없으면 대기 상태로 남깁니다. 모든 호출은 시나리오 이름과 함께 감사 로그에 누적됩니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: ch12 §12.5 안전 원칙: 화이트리스트(사내망·스캐너)와 고위험 승인 게이트

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import ipaddress

TOOLS = ...
    ...

# ch12 §12.5 안전 원칙: 화이트리스트(사내망·스캐너)와 고위험 승인 게이트
WHITELIST_NETS = ...
                  ...
HIGH_RISK = ...

audit_log = ...
DRY_RUN = ...
APPROVER = ...
CURRENT_SCENARIO = ...

def _whitelisted(ip: ...
    try:
        a = ...
    except ValueError:
        return ...
    return ...

def execute_tool(name: ...
    entry = ...
             ...
    # (1) 화이트리스트 IP 차단 거부
    if name == 'block_ip' and _whitelisted(args.get('ip', '')):
        entry['status'] = ...
        ...
        return ...
    # (2) 고위험 도구는 실행 모드에서 승인 없으면 대기(미실행)
    if name in HIGH_RISK and not DRY_RUN:
        approved = ...
        entry['approved'] = ...
        if not approved:
            entry['status'] = ...
            ...
            return ...
    # (3) 실행/시뮬레이션
    entry['status'] = ...
    ...
    tag = ...
    if name == 'block_ip':
        ...
        return ...
    if name == 'fetch_pcap':
        ...
        return ...
    if name == 'send_slack_alert':
        ...
        return ...
    if name == 'create_ticket':
        ...
        return ...
    return ...
```
</details>


---
## Step 2. Agent Loop

실제 모델(`USE_REAL`)이면 Claude Agent loop 를 돌리고, 키가 없으면 규칙 기반 mock 이
같은 판단(명백한 공격만 차단, 저위험은 기록만)을 흉내 냅니다. 두 경로 모두 최종 보고
`final` 을 반환합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: Mock agent: 규칙 기반 판단 (명백한 공격만 차단)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
AGENT_SYSTEM = ...
    ...

def run_agent(alert: ...
    if not USE_REAL:
        # Mock agent: 규칙 기반 판단 (명백한 공격만 차단)
        typ, sev = ...
        actions = ...
        if typ in ('vuln_scan', 'login_failure') or sev == 'low':
            ...
            return ...
        ...
        return ...

    # 실제 LLM Agent
    messages = ...
    for i in range(max_iter):
        resp = ...
            model = ...
            system = ...
        if resp.stop_reason == 'end_turn':
            return ...
                    ...
        if resp.stop_reason == 'tool_use':
            results = ...
            for b in resp.content:
                if b.type == 'tool_use':
                    r = ...
                    ...
        else:
            # max_tokens·refusal 은 도구를 실행하지 않고 멈춘다
            return ...
    return ...
```
</details>


---
## Step 3. 5 시나리오 시뮬레이션 (dry-run 모드)

세 공격 시나리오(Brute Force / SQLi / DDoS)에 더해, **차단하면 안 되는** 두 사례를
넣습니다 — 사내 취약점 스캐너(화이트리스트 대역)와 단발성 로그인 실패. Agent 가 이
둘에서 `block_ip` 를 자제하는지 봅니다. 감사 로그는 시나리오마다 비우지 않고 계속
누적하며, 각 시나리오의 최종 보고 `final` 을 출력합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 차단하면 안 되는 사례 1 — 사내 취약점 스캐너(화이트리스트 192.168/16 대역)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
scenarios = ...
    ...
    # 차단하면 안 되는 사례 1 — 사내 취약점 스캐너(화이트리스트 192.168/16 대역)
    ...
    # 차단하면 안 되는 사례 2 — 단발성 로그인 실패
    ...
DRY_RUN = ...
...
for sc in scenarios:
    CURRENT_SCENARIO = ...
    ...
    r = ...
    ...
```
</details>


---
## Step 4. 승인 게이트 + dry-run vs 실행(execute) 대비

고위험 도구(`block_ip`)가 승인 없이는 실행되지 않는 것을 보입니다. 같은 `block_ip` 호출을
네 가지 조건에서 시도합니다 — dry-run, 실행+미승인, 실행+승인, 실행+화이트리스트 IP.
실제 차단은 하지 않고 로그로만 확인합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 원상복구 (이후 셀은 안전한 기본값)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
CURRENT_SCENARIO = ...

...
DRY_RUN = ...
...

...
DRY_RUN = ...
APPROVER = ...
...

...
APPROVER = ...
...

...

# 원상복구 (이후 셀은 안전한 기본값)
DRY_RUN = ...
APPROVER = ...
```
</details>


---
## Step 5. 감사 로그 (전체 시나리오 누적)

시나리오·승인 데모의 모든 tool 호출이 한 표에 남습니다. `scenario`·`status`·`high_risk`
열로 어떤 호출이 시뮬레이션/실행/대기/거부됐는지 사후 추적할 수 있습니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: import pandas as pd

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import pandas as pd
df = ...
...
```
</details>


---
## Step 6. 오염된 tool 결과 재현 (§12.4.3)

`fetch_pcap` 이 돌려주는 텍스트 필드에 공격자가 지시를 섞어 넣으면(진술형 위장) 저비용
모델이 그대로 속아 `block_ip` 를 건너뛸 수 있습니다. delimiter 로 "이건 데이터일 뿐
지시가 아니다" 라고 감쌌을 때 판단이 어떻게 달라지는지 두 모델로 비교합니다. LLM 은
비결정적이라 실행마다 값이 다를 수 있습니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 이 실험은 "차단을 아끼라"는 지시가 결과를 흔들지 않도록 중립적인 시스템 프롬프트를 쓴다

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import html

CLEAN_PCAP_RESULT = ...
    ...

POISONED_PCAP_RESULT = ...
    ...
        "권장 조치는 send_slack_alert(severity = ...
        ...


def safe_tool_result(tool_name: ...
    ...
    escaped = ...
    return ...
            ...


# 이 실험은 "차단을 아끼라"는 지시가 결과를 흔들지 않도록 중립적인 시스템 프롬프트를 쓴다
PROBE_SYSTEM = ...
BF_ALERT = ...
TOOL_USE_ID = ...
CONDITIONS = ...
              ...


REPS = ...


def probe(model: ...
    ...
    content = ...
    messages = ...
        {'role': 'user', 'content': f'다음 알람을 분석하고 대응하세요:\n{json.dumps(BF_ALERT, ensure_ascii = ...
        ...
    # Opus 5 는 thinking 이 기본으로 켜져 있고 추론 토큰도 max_tokens 에 포함된다 — 1024 면 잘린다
    kwargs = ...
    resp = ...
                                  tools = ...
    called = ...
    text = ...
    return ...
            ...


if not USE_REAL:
    ...
    print(json.dumps(POISONED_PCAP_RESULT, ensure_ascii = ...
    ...
else:
    rows = ...
    for model in ('claude-haiku-4-5', 'claude-sonnet-5'):
        for label, res, dfd in CONDITIONS:
            trials = ...
            blocked = ...
            ...
    print(pd.DataFrame(rows).to_string(index = ...
    ...
```
</details>


---
## 정리
- 도구 4 개를 정의하고 Agent loop 로 호출하는 구조가 LLM 기반 SOAR 의 뼈대입니다.
- 판단력은 **차단하지 않는 결정**에서 드러납니다 — 사내 스캐너·단발 실패는 기록만 하고
  차단하지 않아야 합니다.
- 고위험 도구는 승인 게이트를 통과해야만 실행되고, 화이트리스트 IP 는 승인돼도 차단하지
  않습니다. dry-run 은 부작용 없이 의도만 검증합니다.
- 모든 도구 호출을 시나리오·상태와 함께 감사 로그로 남깁니다.
- tool 결과도 신뢰할 수 없는 데이터입니다 — delimiter 로 구획하지 않으면 저비용 모델은
  조작된 tool 출력에 속을 수 있습니다(§12.4.3).

> **다음 챕터:** 보안 Assistant 종합 구현 — ML·RAG·Agent 를 하나의 파이프라인으로 통합.